# HeartMuLa 3B｜Colab 音樂生成（最精簡版）

使用 Colab 預設 PyTorch、TorchCodec、FFmpeg 6。依序執行下方 **6 個程式格**，直接生成、試聽與下載 MP3，無 WAV 中間檔。

保留先前已確認必要的兩項相容性處理：`transformers==4.57.0`（正確載入 HeartCodec 權重）、TorchTune 的 RoPE cache 修正。**直接 MP3 輸出尚需在 Colab 實測**。

## 1. 安裝套件

In [ ]:
# Download the official source if needed.
# 尚未下載時才取得官方原始碼。
!test -d /content/heartlib || git clone -q https://github.com/HeartMuLa/heartlib.git /content/heartlib

# Install only the dependencies required by the working setup.
# 只補裝已驗證流程所需的相容套件。
%pip -q install "transformers==4.57.0" "tokenizers==0.22.1" "huggingface-hub==0.36.2" "vector-quantize-pytorch==1.27.15"

# Install HeartMuLa without changing Colab's PyTorch stack.
# 安裝 HeartMuLa，不更動 Colab 預裝的 PyTorch。
%pip -q install --no-deps -e /content/heartlib

## 2. 設定歌詞、曲風與長度

In [ ]:
# Manage the input text files and model paths.
# 管理歌詞、曲風與模型路徑。
from pathlib import Path

# Set the lyrics for generation.
# 設定生成歌詞。
LYRICS = """
[Verse]
雨後的街道慢慢亮起燈
微風吹過熟悉的旅程

[Chorus]
讓我們迎著光向前走
把每個故事留在心中
""".strip()

# Set the music style.
# 設定音樂風格。
TAGS = "soft, Sad, Ballad, Longing"

# Set the target duration in seconds.
# 設定音樂長度（秒）。
DURATION = 20

# Save the lyrics for the official generator.
# 儲存歌詞供官方生成程式讀取。
Path("/content/my_lyrics.txt").write_text(LYRICS, encoding="utf-8")

# Save the style tags for the official generator.
# 儲存曲風標籤供官方生成程式讀取。
Path("/content/my_tags.txt").write_text(TAGS, encoding="utf-8")

## 3. 下載官方模型

In [ ]:
# Import the official model downloader.
# 匯入 Hugging Face 模型下載工具。
from huggingface_hub import snapshot_download

# Use one checkpoint directory for all components.
# 使用同一個目錄存放模型元件。
CKPT = Path("/content/heartlib/ckpt")

# Download the tokenizer and generation configuration.
# 下載 tokenizer 與生成設定。
snapshot_download("HeartMuLa/HeartMuLaGen", local_dir=str(CKPT), allow_patterns=["tokenizer.json", "gen_config.json"], max_workers=2)

# Download the 3B music model.
# 下載 3B 音樂模型。
snapshot_download("HeartMuLa/HeartMuLa-oss-3B-happy-new-year", local_dir=str(CKPT / "HeartMuLa-oss-3B"), allow_patterns=["config.json", "*.safetensors", "*.safetensors.index.json"], max_workers=2)

# Download the matching audio decoder.
# 下載對應的音訊 decoder。
snapshot_download("HeartMuLa/HeartCodec-oss-20260123", local_dir=str(CKPT / "HeartCodec-oss"), allow_patterns=["config.json", "*.safetensors", "*.safetensors.index.json"], max_workers=2)

## 4. TorchTune RoPE 相容性修正

先前使用 Colab 預裝 TorchTune 時，曾遇到 `RoPE cache is not built`，導致模型無法開始生成。這格僅在必要時修正 HeartMuLa 的 cache 初始化，**不修改模型權重**。

In [ ]:
# Locate the model source that initializes the attention cache.
# 找出初始化 attention cache 的模型原始碼。
model_file = Path("/content/heartlib/src/heartlib/heartmula/modeling_heartmula.py")

# Read the implementation before applying a compatibility fix.
# 讀取原始碼以準備套用相容性修正。
source = model_file.read_text(encoding="utf-8")

# Match the original cache setup exactly.
# 精確比對官方原始的快取初始化程式。
original = "        with device:\n            self.backbone.setup_caches(max_batch_size, dtype)"

# Build RoPE caches before initializing the attention cache.
# 在 attention cache 初始化前補建 RoPE cache。
replacement = """        with device:
            for module in self.modules():
                if hasattr(module, 'rope_init') and getattr(module, 'is_cache_built', True) is False:
                    module.rope_init()
            self.backbone.setup_caches(max_batch_size, dtype)"""

# Apply the patch only once and fail safely on changed source layouts.
# 只套用一次修正；若官方原始碼變更則安全停止。
if "module.rope_init()" not in source:
    # Avoid editing an unexpected version of the source.
    # 避免修改結構不符的原始碼版本。
    if original not in source:
        # Explain why automatic patching cannot continue.
        # 說明無法繼續自動修正的原因。
        raise RuntimeError("HeartMuLa cache 初始化程式已變動，無法自動套用 RoPE 修正。")
    # Save the compatibility fix to the installed model source.
    # 將相容性修正寫入已安裝的模型原始碼。
    model_file.write_text(source.replace(original, replacement, 1), encoding="utf-8")

## 5. 生成 MP3

In [ ]:
# Run the official generator in a separate Python process.
# 透過獨立 Python 程序執行官方生成程式。
import subprocess
import sys

# Define the final MP3 output path.
# 設定最終 MP3 輸出路徑。
output = Path("/content/music_output.mp3")

# Remove old output so a failed run cannot reuse it.
# 清除舊檔，避免生成失敗時誤用上次的結果。
output.unlink(missing_ok=True)

# Keep the working 3B, FP16, and lazy-load settings.
# 保留先前成功使用的 3B、FP16 與 lazy-load 設定。
command = [
    sys.executable, "-u", "/content/heartlib/examples/run_music_generation.py",
    "--model_path", str(CKPT), "--version", "3B",
    "--lyrics", "/content/my_lyrics.txt", "--tags", "/content/my_tags.txt",
    "--save_path", str(output),
    "--max_audio_length_ms", str(DURATION * 1000),
    "--lazy_load", "true", "--mula_dtype", "float16", "--codec_dtype", "float32",
]

# Generate MP3 directly with the official TorchAudio output path.
# 讓官方程式透過 TorchAudio 直接輸出 MP3。
subprocess.run(command, cwd="/content/heartlib", check=True)

# Reject a missing or empty output.
# 排除未生成或內容為空的輸出。
if not output.is_file() or output.stat().st_size <= 1024:
    # Report the failure before offering a download.
    # 在提供下載前回報失敗。
    raise RuntimeError("MP3 未成功產生，請檢查上方生成紀錄。")

# Show the final MP3 size.
# 顯示最終 MP3 檔案大小。
print(f"MP3 已生成：{output}（{output.stat().st_size:,} bytes）")

## 6. 試聽與下載

In [ ]:
# Import Colab audio playback and download tools.
# 匯入 Colab 音訊試聽與下載工具。
from IPython.display import Audio, display
from google.colab import files

# Preview the generated MP3.
# 在 Notebook 中試聽生成的 MP3。
display(Audio(filename=str(output)))

# Download the MP3 to the local device.
# 將 MP3 下載至本機裝置。
files.download(str(output))